# Data Mining - Lab 7
## Өгөгдлийн агуулах ба OLAP: Roll-up, Drill-down, Slice, Dice

**Dataset:** `fact_sales.csv`, `dim_time.csv`, `dim_product.csv`, `dim_location.csv`

Электрон бараа худалдаалдаг сүлжээ дэлгүүрийн 2024 ба 2025 оны борлуулалтын одон схем (star schema). Нэг fact хүснэгт ба гурван dimension хүснэгтээс бүрдэнэ.

| Хүснэгт | Мөр | Тайлбар |
|---|---|---|
| `fact_sales` | 5332 | Нэг мөр нь нэг захиалгын нэг барааны мөр (grain). `quantity`, `unit_price`, `amount`, `cost` хэмжигдэхүүнтэй |
| `dim_time` | 731 | Өдөр бүрийн мөр: `day`, `month`, `quarter`, `year` |
| `dim_product` | 10 | `product_name`, `brand`, `category`, `list_price` |
| `dim_location` | 10 | `store`, `city`, `region` |

Мөнгөн дүн бүгд долларын (USD) нэгжтэй.

### Lab зорилго
- Өгөгдлийн агуулах (OLAP) ба үйл ажиллагааны систем (OLTP)-ийн ялгааг тайлбарлах
- Одон схемийн fact ба dimension хүснэгтийг холбож, холболтын зөв байдлыг шалгах
- Cuboid, concept hierarchy ойлголтыг кодоор гаргаж, нийт cuboid-ийн тоог тооцох
- Roll-up, drill-down, slice, dice, pivot үйлдлийг pandas-аар хийх
- Хэмжигдэхүүнийг distributive, algebraic, holistic гэж ангилж, дундажийн дундаж гэх мэт алдааг илрүүлэх
- Нэгтгэсэн үр дүнгээр бизнесийн асуултад хариулах

In [21]:
import pandas as pd
import numpy as np

fact = pd.read_csv("fact_sales.csv")
dim_time = pd.read_csv("dim_time.csv")
dim_product = pd.read_csv("dim_product.csv")
dim_location = pd.read_csv("dim_location.csv")

for name, t in [("fact_sales", fact), ("dim_time", dim_time), ("dim_product", dim_product), ("dim_location", dim_location)]:
    print(name, t.shape)
fact.head()

fact_sales (5332, 9)
dim_time (731, 6)
dim_product (10, 5)
dim_location (10, 4)


,sale_id,order_id,time_key,product_key,location_key,quantity,unit_price,amount,cost
0,1,1,20250817,7,10,1,238,238,158
1,2,1,20250817,3,10,1,540,540,476
2,3,1,20250817,1,10,1,650,650,566
3,4,2,20241117,9,2,3,45,135,88
4,5,2,20241117,10,2,2,60,120,80


### TASK 1: Grain ба хэмжигдэхүүн

1. `fact` хүснэгтэд хэдэн мөр, хэдэн ялгаатай `order_id` байна вэ?
5332 мөр, 3300 захиалга.

2. Яагаад мөрийн тоо захиалгын тооноос их байна вэ? (үгээр, grain-тай холбож)
Grain нь "нэг захиалгын нэг барааны мөр". Нэг захиалгад хэд хэдэн бараа орж болдог учраас мөр нь захиалгаас их байна (дунджаар нэг захиалгад 1.6 орчим мөр).

3. `quantity`, `unit_price`, `amount`, `cost` дөрвөөс аль нь нийлбэрлэхэд утга учиртай вэ, аль нь үгүй вэ? (үгээр)
quantity, amount, cost-ыг нэмэхэд утга учиртай. unit_price-ийг нэмэх нь утгагүй, учир нь үнийн нийлбэр гэж юу ч илэрхийлэхгүй.


In [22]:
n_lines = len(fact)
n_orders = fact["order_id"].nunique()
print("мөр:", n_lines, "захиалга:", n_orders)

мөр: 5332 захиалга: 3300


### TASK 2: Foreign key ба холболтын шалгалт

Fact ба dimension-ыг холбохоос өмнө **referential integrity**-г шалгана. Мөн холболт нь мөрийн тоо болон нийт дүнг өөрчлөх ёсгүй (өөрөөр хэлбэл dimension-ы key давхардаагүй).

1. Fact-ийн гурван foreign key (`time_key`, `product_key`, `location_key`) бүр dimension-д байгааг шалга. Тус бүр `True` гарах ёстой.
2. `merge`-ийг `validate="many_to_one"`-тэй хий. Энэ нь dimension дахь key давхардвал алдаа өгнө.
3. Холбосон `dw` хүснэгтийн мөрийн тоо `fact`-тай ижил, `amount`-ийн нийлбэр ижил гэдгийг батал.
4. Холболтыг `validate`-гүйгээр, давхардсан мөртэй dimension-аар хийвэл нийлбэр хэрхэн өөрчлөгдөхийг (fan-out) тайлбарла.
Dimension хүснэгтэд нэг key хоёр мөртэй байвал merge хийхэд fact-ийн мөр бүр хоёр хувь болж давхардана. Тэгэхээр мөрийн тоо, amount-ийн нийлбэр хоёулаа хэтэрхий том гарна (fan-out). `validate="many_to_one"` өгвөл ийм тохиолдолд алдаа өгдөг тул аюулгүй.


In [23]:
print(fact["time_key"].isin(dim_time["time_key"]).all())
print(fact["product_key"].isin(dim_product["product_key"]).all())
print(fact["location_key"].isin(dim_location["location_key"]).all())

dw = (fact
      .merge(dim_time, on="time_key", validate="many_to_one")
      .merge(dim_product, on="product_key", validate="many_to_one")
      .merge(dim_location, on="location_key", validate="many_to_one"))

print(len(dw) == len(fact), dw["amount"].sum() == fact["amount"].sum())
print(dw.columns.tolist())

True
True
True
True True
['sale_id', 'order_id', 'time_key', 'product_key', 'location_key', 'quantity', 'unit_price', 'amount', 'cost', 'date', 'day', 'month', 'quarter', 'year', 'product_name', 'brand', 'category', 'list_price', 'store', 'city', 'region']


## 1) Cuboid ба concept hierarchy

Dimension бүр **concept hierarchy** (нарийвчлалын шат) агуулна.

| Dimension | Шатлал (нарийнаас ерөнхий рүү) | Түвшин |
|---|---|---|
| Time | өдөр, сар, улирал, жил | 4 |
| Product | бүтээгдэхүүн, ангилал | 2 |
| Location | салбар (store), хот (city), бүс (region) | 3 |

Анхаар: `brand` нь шатлалын түвшин **биш**. Apple нь `Утас` ба `Дагалдах хэрэгсэл` хоёуланд байдаг тул brand-аас category руу авирах нь нэг утгатай биш (non-strict). Тиймээс `brand`-ыг энгийн шинж чанар (attribute) гэж үзнэ.

**Cuboid** гэж dimension бүрийн нэг түвшнийг (эсвэл "бүгд" буюу ALL түвшинг) сонгож хийсэн нэгтгэлийг хэлнэ. Хамгийн нарийн cuboid нь **base cuboid**, бүх dimension ALL байвал **apex cuboid** (нэг утга). Dimension i-ийн түвшин Lᵢ бол нийт cuboid-ийн тоо:

T = (L₁ + 1) × (L₂ + 1) × ... × (Lₙ + 1)

### TASK 3: Нийт cuboid-ийн тоо

1. Дээрх гурван dimension-ы түвшний тоогоор нийт cuboid-ийн тоог бод.
2. Хэрэв `Product` dimension-д `brand` түвшин нэмж `бүтээгдэхүүн, brand, ангилал` (3 түвшин) гэж үзвэл тоо хэд болох вэ?
3. Яагаад dimension-ы түвшин нэмэхэд cuboid-ийн тоо хурдан өсдөг вэ? Энэ нь өгөгдлийн агуулахад юуг хэрэгжүүлэхэд хүндрэл учруулдаг вэ? Cuboid-ийн тоо нь түвшин бүрийн (L+1)-ийн үржвэр тул нэг dimension-д түвшин нэмэхэд бусад бүгдтэйгээ үржигдэж хурдан өснө (60 -> 80). Бүгдийг нь урьдчилж бодож хадгалбал зай их эзэлнэ, шинэчлэхэд удаан. Тиймээс заримыг нь л бодож хадгалдаг.


In [24]:
levels = {"time": 4, "product": 2, "location": 3}

# TODO: (L + 1)-ийн үржвэр
total = 1
for L in levels.values():
    total *= (L + 1)
print("нийт cuboid:", total)

levels2 = dict(levels, product=3)
print("brand нэмвэл:", np.prod([L + 1 for L in levels2.values()]))

нийт cuboid: 60
brand нэмвэл: 80


### TASK 4: Таван cuboid-ийг groupby-аар гаргах

Доорх cuboid бүрийг `dw`-ээс гарга. Cuboid бүрийн **эсийн тоо** (мөрийн тоо) ба `amount`-ийн нийлбэрийг хэвлэ.

| Cuboid | Бүлэглэх түвшин |
|---|---|
| Apex | ямар ч dimension-гүй (нийт) |
| 1-D | `year` |
| 2-D | `year`, `category` |
| 3-D | `year`, `category`, `region` |
| Base | `date`, `product_name`, `store` |

Бүх cuboid-ийн нийлбэр ижил гарах ёстой. Яагаад?
Бүх cuboid-ийн нийлбэр 2,519,277 гарсан. Учир нь бүгд ижил мөрүүдийг өөр өөр түвшинд бүлэглэж байгаа, мөр алга болдоггүй. SUM нь distributive тул бүлэглэснээр нийт дүн өөрчлөгдөхгүй.


In [25]:
total_amt = dw["amount"].sum()
cuboids = {
    "apex": [],
    "1-D": ["year"],
    "2-D": ["year", "category"],
    "3-D": ["year", "category", "region"],
    "base": ["date", "product_name", "store"],
}
rows = []
for name, keys in cuboids.items():
    if keys:
        g = dw.groupby(keys)["amount"].sum()
        cells, s = len(g), g.sum()
    else:
        cells, s = 1, total_amt
    rows.append((name, cells, s))
pd.DataFrame(rows, columns=["cuboid", "cells", "sum_amount"])

,cuboid,cells,sum_amount
0,apex,1,2519277
1,1-D,2,2519277
2,2-D,6,2519277
3,3-D,24,2519277
4,base,5041,2519277


## 2) Roll-up

**Roll-up** нь (а) concept hierarchy-аар нарийн түвшнээс ерөнхий түвшин рүү авирах (өдөр -> сар -> улирал -> жил), эсвэл (б) нэг dimension-ыг бүр хасах (dimension reduction) үйлдэл. Roll-up-ийн дараа мөрийн тоо багасах боловч distributive хэмжигдэхүүний (SUM) нийт дүн **өөрчлөгдөхгүй**.

### TASK 5: Time dimension дээр roll-up

Өдрийн түвшнээс эхэлж `amount`-ийг дараах дарааллаар нэгтгэ:

1. өдөр (`date`)
2. сар (`year`, `month`)
3. улирал (`year`, `quarter`)
4. жил (`year`)

Түвшин бүрийн мөрийн тоо болон нийлбэрийг хэвлэ. Нийлбэр бүх түвшинд 2519277 гарах ёстой.

In [26]:
time_levels = {
    "өдөр": ["date"],
    "сар": ["year", "month"],
    "улирал": ["year", "quarter"],
    "жил": ["year"],
}
out = []
for name, keys in time_levels.items():
    g = dw.groupby(keys)["amount"].sum()
    out.append((name, len(g), g.sum()))
pd.DataFrame(out, columns=["түвшин", "мөр", "нийлбэр"])

,түвшин,мөр,нийлбэр
0,өдөр,722,2519277
1,сар,24,2519277
2,улирал,8,2519277
3,жил,2,2519277


### TASK 6: Түгээмэл алдаа: сарыг жилгүйгээр бүлэглэх

Hierarchy-ийн түвшин нь **эцэг түвшингээ агуулсан** түлхүүртэй байх ёстой. Сар `1` нь 2024 оны 1-р сар ч, 2025 оны 1-р сар ч байж болно.

1. `dw.groupby("month")` ба `dw.groupby(["year", "month"])` хоёрын мөрийн тоог харьцуул.
`groupby("month")` 12 мөр, `groupby(["year","month"])` 24 мөр гарна.

2. Эхний аргаар гарсан 1-р сарын нийлбэр нь аль хоёр сарын нийлбэрийг хольсон бэ?
1-р сарын 168,439 нь 2024 оны 1-р сар ба 2025 оны 1-р сарыг хольсон дүн.

3. Хоёр аргын нийт дүнг харьцуул. Алдааг нийт дүнгээр илрүүлж болох уу? (үгээр)
Хоёр аргын нийт дүн ижил (True) тул алдааг нийт дүнгээр илрүүлж болохгүй. Бүлгийн тоо (12 vs 24)-г шалгаж л олно.


In [27]:
wrong = dw.groupby("month")["amount"].sum()
right = dw.groupby(["year", "month"])["amount"].sum()
print(len(wrong), len(right))
print("1-р сар (буруу):", wrong.loc[1])
print("1-р сар 2024 + 2025:", right.loc[(2024, 1)] + right.loc[(2025, 1)])
print(wrong.sum() == right.sum())

12 24
1-р сар (буруу): 168439
1-р сар 2024 + 2025: 168439
True


### TASK 7: Location dimension дээр roll-up

`store` -> `city` -> `region` гурван түвшинд `amount`-ийн нийлбэр ба **мөрийн тоо** (бүлгийн тоо)-г гарга. Хүлээгдэх бүлгийн тоо: 10, 8, 4.

Дараа нь `city` түвшний хүснэгтийг нийлбэрээр нь буурахаар эрэмбэл. Хамгийн их борлуулалттай хот аль вэ?
Бүлгийн тоо 10, 8, 4 гарсан. Хамгийн их борлуулалттай хот нь Улаанбаатар (1,270,629).

In [28]:
loc_levels = {"store": ["store"], "city": ["city"], "region": ["region"]}
for name, keys in loc_levels.items():
    g = dw.groupby(keys)["amount"].sum()
    print(name, len(g), g.sum())

by_city = dw.groupby("city")["amount"].sum().sort_values(ascending=False)
by_city

store 10 2519277
city 8 2519277
region 4 2519277


city
Улаанбаатар    1270629
Эрдэнэт         251916
Дархан          250257
Чойбалсан       218967
Ховд            191830
Баруун-Урт      169467
Мөрөн           126851
Улаангом         39360
Name: amount, dtype: int64

### TASK 8: Roll-up ба dimension reduction

Гурван dimension-ы 3-D cuboid `(year, category, region)`-оос `region`-ыг хасаж `(year, category)` cuboid гарга. Үүнийг:

1. 3-D cuboid-ийн үр дүнг `groupby(...).sum()` хийх замаар (өөрөөр хэлбэл **нэгтгэсэн үр дүнгээс** roll-up)
2. `dw`-ээс шууд `(year, category)`-аар бүлэглэх замаар

хоёр аргаар гарга. Хоёр үр дүн яг ижил гэдгийг `equals` эсвэл `np.allclose`-оор батал. Энэ нь SUM-ийн ямар шинж чанартай холбоотой вэ?
Хоёр арга ижил үр дүн өгсөн (`allclose` = True). SUM нь distributive тул дээд түвшний утгыг доод түвшний нийлбэрүүдээс шууд гаргаж болно. Анхны мөрүүдийг дахин уншихгүй.


In [29]:
cube3 = dw.groupby(["year", "category", "region"])["amount"].sum()
via_cube = cube3.groupby(level=["year", "category"]).sum()
direct = dw.groupby(["year", "category"])["amount"].sum()
print(np.allclose(via_cube, direct))
direct.unstack()

True


category,Дагалдах хэрэгсэл,Ноутбук,Утас
year,,,
2024,175802,361494,645482
2025,209796,432109,694594


## 3) Drill-down

**Drill-down** нь roll-up-ийн урвуу: ерөнхий түвшнээс нарийн түвшин рүү орно (жил -> улирал -> сар, бүс -> хот -> салбар). Бизнесийн шинжилгээнд ихэвчлэн "ерөнхий дүн сонирхолтой байна, яагаад?" гэсэн асуултаар эхэлж нарийн түвшин рүү орно.

### TASK 9: Бүс -> хот -> салбар

1. 2025 оны борлуулалтыг `region`-оор нэгтгэж хамгийн их борлуулалттай бүсийг ол.
2. Тэр бүсийн **доторх** хотуудыг (drill-down) гарга.
3. Тэр хамгийн их хотын **доторх** салбаруудыг гарга.
4. Алхам бүрд өмнөх түвшний нийлбэрээс хэдэн хувийг эзэлж байгааг (`share_pct`) тооц.

Хүлээгдэх хариу: бүс `Төв` (2025 оны нийт борлуулалтын 59.4%), хот `Улаанбаатар`.

In [30]:
d25 = dw[dw["year"] == 2025]

reg = d25.groupby("region")["amount"].sum().sort_values(ascending=False)
top_region = reg.index[0]
print(top_region, round(reg.iloc[0] / reg.sum() * 100, 1), "%")

in_region = d25[d25["region"] == top_region]
cty = in_region.groupby("city")["amount"].sum().sort_values(ascending=False)
cty_tbl = cty.to_frame("amount")
cty_tbl["share_pct"] = (cty_tbl["amount"] / cty_tbl["amount"].sum() * 100).round(1)
print(cty_tbl)

top_city = cty.index[0]
sto = in_region[in_region["city"] == top_city].groupby("store")["amount"].sum().sort_values(ascending=False)
sto

Төв 59.4 %
             amount  share_pct
city                          
Улаанбаатар  663749       83.6
Дархан       130677       16.4


store
Төв салбар       267453
Зайсан салбар    210023
Яармаг салбар    186273
Name: amount, dtype: int64

### TASK 10: Жил -> улирал -> сар

1. 2025 оны улирал бүрийн борлуулалтыг гарга. Хамгийн их улирал аль вэ? Хамгийн их улирал Q4 (426,154)
2. Тэр улирлыг **сар** руу drill-down хий. 
3. Хамгийн их сар аль вэ? Борлуулалтын өсөлтийг юу тайлбарлаж болох вэ? (үгээр, жишээ нь баяр ёслол, шинэ жил)
түүний дотор хамгийн их сар 12-р сар (169,223). Үүнийг шинэ жил, баяр ёслол, бэлэг авах, хямдралтай холбож болох юм. Гэхдээ өгөгдөл өөрөө шалтгааныг батлахгүй.

Хүлээгдэх хариу: улирал Q4, сар 12.

In [31]:
qsum = d25.groupby("quarter")["amount"].sum()
top_q = qsum.idxmax()
print(qsum, "-> хамгийн их:", top_q)

msum = d25[d25["quarter"] == top_q].groupby("month")["amount"].sum()
print(msum, "-> хамгийн их сар:", msum.idxmax())

quarter
1    322782
2    263719
3    323844
4    426154
Name: amount, dtype: int64 -> хамгийн их: 4
month
10     94645
11    162286
12    169223
Name: amount, dtype: int64 -> хамгийн их сар: 12


## 4) Slice, Dice, Pivot

- **Slice**: нэг dimension-ыг **нэг утгаар** тогтоож дэд cube авна (жишээ нь `year == 2025`).
- **Dice**: хоёр ба түүнээс олон dimension дээр **утгын олонлог** тогтооно.
- **Pivot (rotate)**: тэнхлэгийг сольж харагдах байдлыг өөрчилнө (мөр, баганыг сольж болно).

### TASK 11: Slice ба dice

1. **Slice**: `year == 2025`. Нийлбэрийг гарга, 1336499 байх ёстой.
2. **Dice**: 2025 он, `region` нь `Төв` эсвэл `Баруун`, `category` нь `Утас` эсвэл `Ноутбук`, `quarter` нь 3 эсвэл 4. Мөрийн тоо ба нийлбэрийг гарга.
3. Slice ба dice хоёр хэдэн dimension дээр нөхцөл тавьдгаараа ялгагдах вэ? Slice нэг dimension дээр нэг утга тогтооно, dice хоёр ба түүнээс олон dimension дээр утгын олонлог тогтооно.


Хүлээгдэх хариу (2): 605 мөр, нийлбэр 441766.

In [32]:
sl = dw[dw["year"] == 2025]
print("slice:", sl["amount"].sum())

dc = dw[
    (dw["year"] == 2025)
    & (dw["region"].isin(["Төв", "Баруун"]))
    & (dw["category"].isin(["Утас", "Ноутбук"]))
    & (dw["quarter"].isin([3, 4]))
]
print("dice:", len(dc), dc["amount"].sum())

slice: 1336499
dice: 605 441766


### TASK 12: Pivot ба нийт дүн (margins)

1. 2025 оны өгөгдлөөр мөрөнд `region`, баганад `quarter`, утгад `amount`-ийн нийлбэр бүхий pivot хүснэгт гарга. `margins=True`, `margins_name="Нийт"` ашигла.
2. Баруун-доод буланг (grand total) 2025 оны нийт дүнтэй тулга.
3. Тэнхлэгийг сольж (`region` баганад, `quarter` мөрөнд) дахин гарга. Энэ нь pivot үйлдэл мөн үү?
Тийм, мөр баганыг солих нь pivot (rotate) үйлдэл. Мэдээлэл өөрчлөгдөхгүй, зөвхөн харагдах байдал солигдоно.
4. Margin дахь нийт нь аль cuboid-ийн утга вэ? Мөрийн "Нийт" нь quarter cuboid, баганын "Нийт" нь region cuboid, баруун-доод булан нь apex cuboid (2025 оны нийт).


In [33]:
pv = pd.pivot_table(
    d25, index="region", columns="quarter", values="amount",
    aggfunc="sum", margins=True, margins_name="Нийт"
)
print(pv)
print(pv.loc["Нийт", "Нийт"] == d25["amount"].sum())
pv.T

quarter       1       2       3       4     Нийт
region                                          
Баруун    33926   24097   35159   50779   143961
Зүүн      53441   33653   53955   61615   202664
Төв      189007  164480  193839  247100   794426
Хангай    46408   41489   40891   66660   195448
Нийт     322782  263719  323844  426154  1336499
True


region,Баруун,Зүүн,Төв,Хангай,Нийт
quarter,,,,,
1,33926,53441,189007,46408,322782
2,24097,33653,164480,41489,263719
3,35159,53955,193839,40891,323844
4,50779,61615,247100,66660,426154
Нийт,143961,202664,794426,195448,1336499


### TASK 13: Сул cube: хоосон эс ба тэг

Улаангом салбар 2025 оны 3-р сараас нээгдсэн. Тиймээс 2024 онд нэг ч мөр байхгүй.

1. `store` мөрөнд, `year` баганад `amount`-ийн нийлбэр pivot гарга. Улаангомын 2024 дээр юу харагдаж байна вэ (`NaN` эсвэл `0`)? Улаангомын 2024 дээр NaN гарсан.
2. `fill_value=0`-ээр хоосон эсийг нөх. Хоосон ("мэдээлэл байхгүй") ба тэг ("борлуулалт 0") хоёрын утга учир ялгаатай эсэхийг бодож үз.
Хоосон нь "мэдээлэл байхгүй" (салбар тэр үед нээгдээгүй), тэг нь "байсан ч зарсангүй" гэсэн утгатай. 0-ээр нөхвөл дундаж гэх мэтийг буруу болгож болно.
3. Улаангомын хамгийн эхний борлуулалтын огноог ол.
Хамгийн эхний борлуулалт 2025-03-06.

Хүлээгдэх хариу (3): 2025-03-06.

In [34]:
sy = pd.pivot_table(dw, index="store", columns="year", values="amount", aggfunc="sum")
print(sy)
print(sy.fillna(0).sum().sum() == dw["amount"].sum())

print("эхний борлуулалт:", dw.loc[dw["store"] == "Улаангом салбар", "date"].min())

year                   2024      2025
store                                
Баруун-Урт салбар   71966.0   97501.0
Дархан салбар      119580.0  130677.0
Зайсан салбар      195802.0  210023.0
Мөрөн салбар        59644.0   67207.0
Төв салбар         244459.0  267453.0
Улаангом салбар         NaN   39360.0
Ховд салбар         87229.0  104601.0
Чойбалсан салбар   113804.0  105163.0
Эрдэнэт салбар     123675.0  128241.0
Яармаг салбар      166619.0  186273.0
True
эхний борлуулалт: 2025-03-06


## 5) Хэмжигдэхүүний төрөл

Хэмжигдэхүүнийг нэгтгэх функцээр нь гурван ангилна:

| Төрөл | Тайлбар | Жишээ |
|---|---|---|
| Distributive | Хэсэг бүрийн үр дүнгээс бүгдийг шууд гаргаж болно | SUM, COUNT, MIN, MAX |
| Algebraic | Хязгаарлагдмал тооны distributive утгаас гарна | AVG = SUM / COUNT |
| Holistic | Хэсэгчилсэн үр дүнгээс гаргах боломжгүй | MEDIAN, MODE, RANK |

Cube-д roll-up хийхэд distributive ба algebraic хэмжигдэхүүнийг доод түвшний нэгтгэлээс бодож болдог. Holistic-ийг бүр доод түвшний өгөгдлөөс дахин бодох хэрэгтэй.

### TASK 14: Дундажийн дундаж

2024 оны **нэг борлуулалтын мөрийн дундаж `amount`**-ийг хоёр аргаар бод:

1. Зөв арга: 2024 оны бүх мөрийн `amount`-ийн дундаж.
2. Буруу арга: улирал бүрийн дундажийг олоод тэдгээрийн дундажийг авах.
3. Хоёр үр дүн яагаад ялгаатай вэ? Улирал бүрийн мөрийн тоо ижил биш. Дундажуудын дундаж нь цөөн мөртэй улирлыг их мөртэй улирлаар адил жинлэчихдэг.

4. Зөв аргыг cube-ээс гаргахын тулд улирал бүрт ямар **хоёр** distributive утгыг хадгалах хэрэгтэй вэ? Тэдгээрээр жилийн дундажийг гарга.
Улирал бүрт SUM ба COUNT хадгална. Жилийн дундаж = SUM-уудын нийлбэр / COUNT-уудын нийлбэр = 471.41.


In [35]:
y24 = dw[dw["year"] == 2024]

true_avg = y24["amount"].mean()
by_q = y24.groupby("quarter")["amount"].agg(["mean", "sum", "count"])
wrong_avg = by_q["mean"].mean()
print(round(true_avg, 2), round(wrong_avg, 2))

# algebraic: SUM ба COUNT-оос
alg_avg = by_q["sum"].sum() / by_q["count"].sum()
print(round(alg_avg, 2), np.isclose(alg_avg, true_avg))

471.41 471.91
471.41 True


### TASK 15: Median нь holistic

2024 оны мөр бүрийн `amount`-ийн медианыг:

1. бүх мөрөөс шууд (зөв)
2. улирал бүрийн медианы медианаар (буруу)

бод. Эхний аргаар гарсан утга хоёр дахь аргаас ялгаатай юу? Median-ыг SUM шиг cube-ийн дээд түвшинд нэгтгэж болохгүй гэдгийг тайлбарла.
Ялгаатай. Медиан нь дунд байгаа утга тул хэсэг бүрийн медианаас нийтийнхийг гаргах томьёо байхгүй (holistic). SUM шиг дээд түвшин рүү нэгтгэж болохгүй, анхны мөрөөс дахин бодох хэрэгтэй.


Хүлээгдэх хариу: зөв 238.0, буруу 230.0.

In [36]:
true_med = y24["amount"].median()
q_med = y24.groupby("quarter")["amount"].median()
wrong_med = q_med.median()
print(true_med, wrong_med, true_med == wrong_med)

238.0 230.0 False


### TASK 16: Нэмж болохгүй хэмжигдэхүүн: нэгж үнэ

`unit_price` нь **additive биш** (non-additive) хэмжигдэхүүн: үнийг нийлүүлж нэмэх нь утгагүй.

1. Category бүрийн `unit_price`-ийн энгийн дундаж (`mean`)-ийг гарга.
2. Category бүрийн **жигнэсэн дундаж үнэ** `SUM(amount) / SUM(quantity)`-ийг гарга.
3. Хоёр үр дүн яагаад ялгаатай вэ? Аль нь "нэг ширхэг бараа дунджаар хэдэнд зарагдсан" гэсэн асуултад зөв хариулт вэ?
Энгийн дундаж нь мөр бүрийг адил жинтэй авдаг, хэдэн ширхэг зарагдсаныг тооцдоггүй. Жигнэсэн дундаж SUM(amount)/SUM(quantity) нь тооцдог. "Нэг ширхэг бараа дунджаар хэдэнд зарагдсан" гэсэн асуултад жигнэсэн нь зөв.


In [37]:
g = dw.groupby("category")
simple = g["unit_price"].mean()
weighted = g["amount"].sum() / g["quantity"].sum()

pd.DataFrame({"энгийн": simple.round(2), "жигнэсэн": weighted.round(2)})

,энгийн,жигнэсэн
category,,
Дагалдах хэрэгсэл,90.89,91.96
Ноутбук,662.21,661.89
Утас,631.20,627.67


## 6) Бизнесийн шинжилгээ

### TASK 17: Ашиг ба ашгийн хувь

`profit = amount - cost`, `margin_pct = SUM(profit) / SUM(amount) * 100`.

1. `dw`-д `profit` багана нэм.
2. `category` тус бүрийн борлуулалт, ашиг, `margin_pct`-ийг гарга.
3. Борлуулалт хамгийн их category ба ашгийн хувь хамгийн өндөр category нэг үү?
Үгүй. Борлуулалт хамгийн их нь Утас, ашгийн хувь хамгийн өндөр нь Дагалдах хэрэгсэл (35.1%).
4. `margin_pct`-ийг category-оор roll-up хийхдээ яагаад мөр бүрийн хувийн дундажийг биш, нийлбэрүүдийн харьцааг ашиглах ёстой вэ? Хувь нь харьцаа учраас том жижиг мөрийг адил жинтэй авч дундажлавал буруу гарна. Тиймээс SUM(profit) / SUM(amount) гэж нийлбэрүүдийн харьцааг авна.

Хүлээгдэх хариу: борлуулалт хамгийн их `Утас`, ашгийн хувь хамгийн өндөр `Дагалдах хэрэгсэл` (35.1%).

In [38]:
dw["profit"] = dw["amount"] - dw["cost"]
cat = dw.groupby("category").agg(amount=("amount", "sum"), profit=("profit", "sum"))
cat["margin_pct"] = (cat["profit"] / cat["amount"] * 100).round(1)
cat.sort_values("amount", ascending=False)

,amount,profit,margin_pct
category,,,
Утас,1340076,134622,10.0
Ноутбук,793603,95705,12.1
Дагалдах хэрэгсэл,385598,135224,35.1


### TASK 18: Жилээс жилийн өсөлт (YoY)

1. Улирал бүрийн `amount`-ийг жилээр нь баганад гаргасан pivot хий (`index="quarter"`, `columns="year"`).
2. `growth_pct = (2025 / 2024 - 1) * 100` багана нэм.
3. Аль улиралд өсөлт хамгийн их, аль улиралд хамгийн бага вэ?
Өсөлт хамгийн их нь Q3 (22.5%), хамгийн бага нь Q2 (3.3%). Өсөлтийн нэг хэсэг нь 2025 онд нээгдсэн Улаангом салбараас.


Хүлээгдэх хариу: хамгийн их өсөлт Q3 (22.5%), Q2 дээр 3.3%.

In [39]:
yq = dw.pivot_table(index="quarter", columns="year", values="amount", aggfunc="sum")
yq["growth_pct"] = ((yq[2025] / yq[2024] - 1) * 100).round(1)
yq

year,2024,2025,growth_pct
quarter,,,
1,305587,322782,5.6
2,255175,263719,3.3
3,264345,323844,22.5
4,357671,426154,19.1


### TASK 19: Top-N ба бүсийн эзлэх хувь

1. 2025 оны борлуулалтаар хамгийн их 3 салбарыг `nlargest`-ээр ол.
2. Бүс бүрийн эзлэх хувийг (`share_pct`) нийт 2025 оны борлуулалтад харьцуулан гарга. Хувийн нийлбэр 100 байх ёстой.
Бүсийн хувь: Төв 59.4%, Зүүн 15.2%, Хангай 14.6%, Баруун 10.8%
3. Хэрэв нэг бүсийн хувь хэт өндөр бол бизнесийн ямар эрсдэл гарч болох вэ? 
Нэг бүсээс хэт хамаарвал тэр бүсэд эдийн засгийн хямрал, өрсөлдөгч, түгээлтийн асуудал гарахад нийт орлого огцом буурна.

Хүлээгдэх хариу (1): Төв салбар, Зайсан салбар, Яармаг салбар.

In [40]:
store25 = d25.groupby("store")["amount"].sum()
print(store25.nlargest(3))

share = (d25.groupby("region")["amount"].sum() / d25["amount"].sum() * 100).round(1)
print(share.sort_values(ascending=False))
print(round(share.sum(), 1))

store
Төв салбар       267453
Зайсан салбар    210023
Яармаг салбар    186273
Name: amount, dtype: int64
region
Төв       59.4
Зүүн      15.2
Хангай    14.6
Баруун    10.8
Name: amount, dtype: float64
100.0


## 7) Мини төсөл: OLAP тайлан

### TASK 20: Менежерт зориулсан таван асуулт

Доорх таван асуултад кодоор хариулж, хариу бүрийн доор cuboid-ийн нэр (ямар түвшин, ямар dimension) болон ашигласан OLAP үйлдлийг (roll-up, drill-down, slice, dice, pivot) нэрлэ.

1. Хоёр жилд нийт хэдэн доллар борлуулсан бэ? Жил бүрийнх хэд вэ?
**Нийт 2,519,277; 2024 онд 1,182,778; 2025 онд 1,336,499.** Cuboid: apex ба 1-D (year). Үйлдэл: roll-up (time-ийг жил хүртэл).
2. 2025 онд аль category нь жилээс жилд хамгийн их өссөн бэ (`amount`-ийн утгаар)?
**Хамгийн их өссөн category: Ноутбук (+70,615).** Cuboid: 2-D (year, category). Үйлдэл: roll-up + pivot.
3. Хамгийн их борлуулалттай бүсийн хамгийн сул салбар аль вэ?
**Хамгийн их бүс Төв, түүний хамгийн сул салбар Дархан салбар (130,677).** Cuboid: (year, region), дараа нь (year, store). Үйлдэл: slice (2025), drill-down (region -> store).
4. Өдрийн түвшинд хамгийн их нэг өдрийн борлуулалттай өдөр аль вэ? Тэр өдрийг (сар, улирал) түвшинд тавьж хар.
**Хамгийн их өдөр 2024-12-28 (17,733).** Өдрийн түвшин нь time-ийн base түвшин. Сар руу roll-up хийхэд 2024 оны 12-р сар, улирал руу Q4. Үйлдэл: roll-up (өдөр -> сар -> улирал).
5. Нэг захиалгад дунджаар хэдэн ширхэг бараа (`quantity`-ийн нийлбэр) байна вэ? Энэ хэмжигдэхүүн ямар төрөлд (distributive, algebraic, holistic) ордог вэ?
**Нэг захиалгад дунджаар 2.28 ширхэг бараа.** Төрөл: algebraic, учир нь SUM(quantity) / захиалгын тоо гэсэн харьцаа.


In [41]:
# 1
print(dw.groupby("year")["amount"].sum(), dw["amount"].sum())

# 2
cy = dw.pivot_table(index="category", columns="year", values="amount", aggfunc="sum")
cy["change"] = cy[2025] - cy[2024]
print(cy.sort_values("change", ascending=False))

# 3
top_reg = d25.groupby("region")["amount"].sum().idxmax()
print(d25[d25["region"] == top_reg].groupby("store")["amount"].sum().sort_values().head(1))

# 4
day = dw.groupby("date")["amount"].sum()
print(day.idxmax(), day.max())

# 5
per_order = dw.groupby("order_id")["quantity"].sum()
print(round(per_order.mean(), 2))

year
2024    1182778
2025    1336499
Name: amount, dtype: int64 2519277
year                 2024    2025  change
category                                 
Ноутбук            361494  432109   70615
Утас               645482  694594   49112
Дагалдах хэрэгсэл  175802  209796   33994
store
Дархан салбар    130677
Name: amount, dtype: int64
2024-12-28 17733
2.28


### Тайлбар
-  

## 8) Дүгнэлт асуултууд (Concept Check)

### TASK 21: Verbal

1. OLTP ба OLAP-ийн гол ялгааг гурван шинж чанараар тайлбарла.
**OLTP** нь өдөр тутмын захиалга, төлбөр, олон богино insert/update хийдэг, өгөгдөл нь normalized, зөвхөн одоогийн мэдээлэл агуулна. **OLAP** нь шинжилгээний систем, том унших query хийдэг, олон жилийн түүхэн өгөгдөлтэй, бүтэц нь одон схем маягийн.
2. Одон схем ба цасан ширхэг (snowflake) схемийн ялгаа юу вэ? Яагаад өгөгдлийн агуулахад одон схемийг ихэвчлэн илүүд үздэг вэ?
**Одон схемд** dimension бүр нэг хүснэгт, fact-ийг шууд тойрно. **Цасан ширхэгт** dimension нь normalize хийгдэж хэд хэдэн хүснэгтэд хуваагдана. Одон схемд join цөөн тул query хурдан, бүтэц энгийн учраас илүүд үздэг.
3. Roll-up ба drill-down-ийн ялгааг `Улаангом` салбарын жишээгээр тайлбарла.
**Roll-up** нь Улаангом салбараас Улаангом хот, түүнээс Баруун бүс рүү ерөнхий болгоно. **Drill-down** нь Баруун бүсээс Улаангом хот, Улаангом салбар руу нарийвчилна.
4. Slice ба dice-ийг нэг өгүүлбэрээр ялга.
**Slice** нэг dimension-ыг нэг утгаар тогтооно, **dice** хоёр ба түүнээс олон dimension дээр олон утга сонгоно.

5. Яагаад `AVG`-ийг roll-up-д хэсэг бүрийн дундажаар биш, `SUM` ба `COUNT`-оор бодох ёстой вэ? Median-ы хувьд яагаад боломжгүй вэ?
Хэсэг бүрийн мөрийн тоо ижил биш тул дундажуудын дундаж буруу гарна (471.91, зөв нь 471.41). SUM ба COUNT хадгалаад нийлбэрийг нийлбэрт нь хуваавал зөв болно. Медианд ийм томьёо байхгүй (230, зөв нь 238), тиймээс анхны мөрөөс дахин бодно.
6. Cuboid-ийн тоог бүгдийг нь урьдчилж тооцох (full materialization) ба зөвхөн зарим нэгийг тооцох (partial materialization)-ын давуу, сул талыг харьцуул.
**Full** нь бүх cuboid-ийг урьдчилж бодсон тул query маш хурдан, гэхдээ зай их эзэлж шинэчлэхэд удаан. Partial нь зарим cuboid-ийг л хадгалдаг тул зай хэмнэнэ, гэхдээ хадгалаагүй cuboid-ийн query арай удаан. Практикт ихэвчлэн partial ашигладаг.

# Data Mining - Lab 7 Тайлан
## Өгөгдлийн агуулах ба OLAP: Roll-up, Drill-down

**Нэр: Буяндэлгэр**  
**Оюутны код: s22c008b**  
**Анги / Бүлэг: КУ5**  
**Огноо: 2026 11.5**  

---

## 1. Lab зорилго (1-2 өгүүлбэр)
Одон схемийн хүснэгтүүдийг холбоод roll-up, drill-down, slice, dice, pivot хийж сурах. Мөн дундаж, медиан зэргийг cube дээр нэгтгэхэд ямар алдаа гардгийг харах.

---

## 2. Dataset-ийн товч тайлбар
- Fact хүснэгтийн grain: нэг захиалгын нэг барааны мөр
- Dimension хүснэгтүүд ба hierarchy: Time (өдөр, сар, улирал, жил), Product (бүтээгдэхүүн, ангилал), Location (салбар, хот, бүс)
- Fact-ийн мөрийн тоо: 5332 мөр, 3300 захиалга

---

## 3. Одон схем ба холболт
- Referential integrity-г хэрхэн шалгасан бэ: Fact-ийн time_key, product_key, location_key бүрийг dimension хүснэгтийн key-тэй isin().all()-аар шалгасан, гурав бүгд True гарсан. Мөн merge хийхдээ validate="many_to_one" өгсөн тул dimension дээр key давхардсан бол алдаа өгөх байсан.
- Холболтын дараа мөрийн тоо, нийлбэр хадгалагдсан уу: Тийм. Холбосны дараа мөр 5332 хэвээр, amount-ийн нийлбэр 2,519,277 хэвээр байсан.

---

## 4. Cuboid
| Cuboid | Эсийн тоо | Нийлбэр |
|---|---|---|
| Apex | 1 | 2,519,277 |
| 1-D (year) | 2 | 2,519,277 |
| 2-D (year, category) | 6 | 2,519,277 |
| 3-D (year, category, region) | 24 | 2,519,277 |
| Base | 5041 | 2,519,277 |

- Нийт cuboid-ийн тоо ба тооцоолсон арга: Dimension бүрийн түвшний тоонд 1 (ALL түвшин) нэмээд үржүүлнэ. (4+1) × (2+1) × (3+1) = 5 × 3 × 4 = 60. Product-д brand түвшин нэмбэл 5 × 4 × 4 = 80 болно. Бүх cuboid дээр нийлбэр ижил гарсан, учир нь SUM нь distributive бөгөөд ижил мөрүүдийг зөвхөн өөр түвшинд бүлэглэж байгаа.

---

## 5. OLAP үйлдлүүд
| Үйлдэл | Ашигласан түвшин, нөхцөл | Гол үр дүн |
|---|---|---|
| Roll-up (time) | өдөр -> сар -> улирал -> жил | 722 -> 24 -> 8 -> 2 мөр, нийлбэр бүгд 2,519,277 |
| Roll-up (location) | салбар -> хот -> бүс | 10 -> 8 -> 4 бүлэг, хамгийн их хот Улаанбаатар (1,270,629) |
| Drill-down (region -> city -> store) | 2025 он, Төв бүс -> Улаанбаатар -> салбар | Төв бүс 59.4%, Улаанбаатар бүсийн 83.6%, хамгийн их салбар Төв салбар (267,453). Мөн 2025 онд Q4 хамгийн их, түүний дотор 12-р сар |
| Slice | year = 2025 | Нийлбэр 1,336,499 |
| Dice | 2025, Төв/Баруун бүс, Утас/Ноутбук, Q3/Q4 | 605 мөр, нийлбэр 441,766 |
| Pivot | region x quarter, margins=True | Нийт дүн 1,336,499, тэнхлэг солиход ч ижил |

---

## 6. Хэмжигдэхүүний төрөл
| Хэмжигдэхүүн | Төрөл | Зөв тооцоолох арга |
|---|---|---|
| SUM(amount) | Distributive | Хэсэг бүрийн нийлбэрийг нэмнэ |
| AVG(amount) | Algebraic | SUM / COUNT. Улирлын дундажийн дундаж буруу (471.91, зөв нь 471.41) |
| MEDIAN(amount) | Holistic | Анхны мөрөөс дахин бодно. Улирлын медианы медиан буруу (230, зөв нь 238) |
| margin_pct | Algebraic (харьцаа) | SUM(profit) / SUM(amount). Мөр бүрийн хувийн дундаж биш |

---

## 7. Бизнесийн дүгнэлт
1. Борлуулалт 2024 онд 1,182,778 байснаа 2025 онд 1,336,499 болж 13% өссөн. Гэхдээ өсөлтийн нэг хэсэг нь 2025 онд нээгдсэн Улаангом салбараас (39,360) ирсэн. Чойбалсан салбар цорын ганц буурсан салбар байна.
2. Төв бүс 2025 оны борлуулалтын 59.4%-ийг, Улаанбаатар нь Төв бүсийн 83.6%-ийг бүрдүүлж байна. Нэг бүс, нэг хотоос хэт хамааралтай байгаа нь эрсдэлтэй.
3. Утас хамгийн их зарагддаг ч ашгийн хувь нь 10.0%. Дагалдах хэрэгсэл бага зарагдах ч 35.1%-ийн ашигтай, нийт ашиг нь (135,224) Утасныхаас (134,622) арай их байна.

---

## 8. Reflection

1. Яагаад сарыг жилгүйгээр бүлэглэх нь алдаа вэ? Сар 1 нь 2024 болон 2025 оны 1-р сарыг хольж нэг бүлэгт хийдэг. Нийт дүн ижил гардаг тул дүнгээр нь алдааг олохгүй, зөвхөн бүлгийн тоо (12, 24)-оор мэдэгдэнэ.
2. Энэ lab өмнөх data preparation lab-уудтай ямар холбоотой вэ? Өмнөх lab-уудад key шалгах, merge хийх, хоосон утгатай ажиллах зэргийг хийсэн. Энд ч адилхан, холболт буруу бол бүх тооцоо буруу гарна. Мөн Улаангомын 2024 оны NaN-г 0-тэй андуурч болохгүй гэдэг нь missing value-тай ажилладагтай төстэй.
3. Ямар хэсэг хамгийн ойлгомжгүй байсан бэ? Median яагаад cube дээр нэгтгэгддэггүй байсан нь надад хамгийн төвөгтэй санагдсан. Дундаж нь SUM ба COUNT-оор гардаг ч медианд тийм томьёо байхгүй.